In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!cp -r /content/drive/MyDrive/data /content/

# SHAP Feature Importance & MOA-stratified Analysis

## Overview
This notebook builds on the fitted models from notebook 02 and performs
two complementary analyses:

### Part A - MOA Annotation
Annotate CTRPv2 compounds with mechanism of action (MOA) via a **multi-source
waterfall**, then collapse the raw MOA strings into coarse classes for
per-class comparison.

Sources are tried in order of decreasing reliability, first hit wins:
1. **ChEMBL by ChEMBL ID** (missing IDs are first backfilled via PubChem
   cross-reference, since PharmacoDB only supplies ChEMBL IDs for ~35% of
   compounds but PubChem CIDs for ~68%)
2. **ChEMBL by InChIKey** (structure-based lookup, for compounds ChEMBL
   knows but PharmacoDB didn't link)
3. **PRISM Repurposing Hub** (Broad Institute), fetched via the **Figshare
   API**. Matched both by
   compound name/synonyms and, for compounds named by their internal
   **BRD-Kxxxxxxxx** ID, directly against PRISM's own **IDs** column
4. **PubChem MeSH Pharmacological Classification**
5. **Manual compound-name rules** (curated one-off mappings for specific
   compounds ChEMBL/PRISM/PubChem don't resolve)
6. **INN/USAN name-suffix heuristic**
Raw MOA text (wherever it was found) is mapped to one of ~11 coarse,
interpretable classes via keyword rules (**MOA_RULES** / **COMPOUND_NAME_RULES**),
guarded by a small smoke test so future rule edits can't silently break
previously-correct classifications. Compounds the waterfall can't resolve
are exported to a CSV for **manual review** (with auto-generated PubChem/
ChEMBL/Google search links), and manually-assigned labels are merged back
in. Classes with fewer than **MIN_CLASS_SIZE** (5) compounds are folded
into **Other/Unknown** **after** manual review, not before, so a genuinely
resolved but small class isn't discarded before a human gets to see it.
Each compound's final row keeps full provenance (**moa_source**,
**mechanism_raw_extended**) alongside the class, for future auditing.

### Part B - SHAP Analysis
For each modeled compound, compute SHAP values using the fitted models
from notebook 02. Two levels:
1. **Per-gene SHAP** - which specific genes/proteins drive predictions?
2. **Per-layer SHAP** - which omics layer (expression / mutations / CNV)
   is most informative per drug class?
### Part C - MOA-stratified Comparison
Group compounds by MOA class and compare omics-layer importance profiles.
Export top-gene lists per MOA class as a future input for KEGG/GO analysis.

## Key design decisions
- **Best model from notebook 02** is read from **shap_handover_recommendation.csv**
- **MOA sourced from a multi-source waterfall, not ChEMBL alone** - ChEMBL
  ID coverage from PharmacoDB is only ~35%; PubChem cross-reference
  backfill, PRISM Repurposing Hub (via Figshare), PubChem MeSH, manual
  rules, and an INN-suffix heuristic together raise resolved coverage far
  above the ChEMBL-only baseline (baseline vs. final counts are printed in
  Part A for comparison)
- **Manual review happens before the small-class merge**, not after, so
  small-but-real automatically-resolved classes remain available as valid
  manual-label targets instead of being silently pre-collapsed into
  **Other/Unknown**
- **Default SHAP mode: **single_omics** per layer** - each layer has its own Ridge model,
  gene names are clean (no prefix). For cross-layer comparison, importance is
  normalized across layers: **layer_importance[layer] = sum(|SHAP|) / sum(all |SHAP|)**.
- **Also supports **early_fusion**** - single model, features prefixed **layer::gene**.
- **Explainer selection** (per model type):
  - **RidgeCV**, **ElasticNet**, **RidgeSelectK** -> **shap.LinearExplainer**
  - **XGBoost**, **LightGBM** -> **shap.TreeExplainer**
- **Fold averaging** - SHAP is computed for each of the 5 saved fold-models and
  averaged. Simpler than out-of-fold but appropriate for feature importance (not for
  prediction quality assessment).
- **Caching** - SHAP results (and each MOA data source: ChEMBL, PRISM, PubChem
  MeSH) are saved to parquet/CSV; re-run is instant, and stale caches are
  invalidated automatically when they would otherwise mask new PubChem
  cross-reference resolutions.

## 0. Setup

In [ ]:
%pip install -q shap pandas numpy matplotlib seaborn pyarrow joblib requests tqdm

In [ ]:
import os
import re
import time
import warnings
from pathlib import Path
from typing import Optional

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import requests
from tqdm.auto import tqdm
from io import StringIO

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

try:
    import shap
    print(f"shap {shap.__version__}")
except ImportError:
    raise ImportError("Install shap: %pip install shap")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110

PROJECT_DIR = Path.cwd()

In [ ]:
DATA_DIR      = PROJECT_DIR / "data"
RAW_DIR       = DATA_DIR / "raw"
SHAP_DIR      = DATA_DIR / "shap_results"
SHAP_DIR.mkdir(parents=True, exist_ok=True)

print("Imports OK")
print(f"PROJECT_DIR: {PROJECT_DIR}")

In [ ]:
# Load handover from notebook 02
# Find the most recent results directory (any LAYER_CONFIG config)
def find_results_dir(data_dir: Path) -> Path:
    """Find the most recent model_results directory."""
    handover_files = list(data_dir.glob("model_results_*/shap_handover_recommendation.csv"))
    if not handover_files:
        raise FileNotFoundError(
            "No shap_handover_recommendation.csv found.\n"
            "Run notebook 02 first to train and save models."
        )
    # Prefer the one matching current LAYER_CONFIG if multiple exist
    handover_files.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    return handover_files[0].parent

RESULTS_DIR = find_results_dir(DATA_DIR)
handover_path = RESULTS_DIR / "shap_handover_recommendation.csv"
handover = pd.read_csv(handover_path).iloc[0]

MODEL_DIR     = Path(handover["models_dir"])
PROCESSED_DIR = Path(handover["processed_dir"])
ACTIVE_LAYERS = [l.strip() for l in handover["active_layers"].split(",")]

BEST_MODEL_FAMILY = handover["model_family"]
BEST_STRATEGY     = handover["strategy"]
BEST_OMICS_LAYER  = handover["omics_layer"]
BEST_FEATURE_KEY  = handover["feature_key"]
N_SPLITS          = 5

print(f"RESULTS_DIR   : {RESULTS_DIR}")
print(f"MODEL_DIR     : {MODEL_DIR}")
print(f"PROCESSED_DIR : {PROCESSED_DIR}")
print(f"ACTIVE_LAYERS : {ACTIVE_LAYERS}")
print(f"\nBest config from notebook 02:")
print(f"  model_family  : {BEST_MODEL_FAMILY}")
print(f"  strategy      : {BEST_STRATEGY}")
print(f"  omics_layer   : {BEST_OMICS_LAYER}")
print(f"  feature_key   : {BEST_FEATURE_KEY}")
print(f"  mean_pearson  : {handover['mean_pearson']:.4f}")
print(f"  SHAP explainer: {handover['shap_explainer']}")

In [ ]:
SHAP_MODE = "best"   # "single_omics" | "early_fusion" | "best"

if SHAP_MODE == "best":
    SHAP_MODE = "early_fusion" if BEST_STRATEGY == "early_fusion" else "single_omics"

print(f"SHAP_MODE: {SHAP_MODE}")

N_BACKGROUND_SAMPLES = 100    # samples for LinearExplainer background
MAX_COMPOUNDS        = None   # None = all eligible
TOP_N_GENES_PER_MOA  = 30     # genes exported per MOA class for KEGG/GO

## Part A - MOA Annotation

Goal: assign every CTRPv2 compound a coarse mechanism-of-action (MOA) class, using a multi-source waterfall (ChEMBL -> PRISM Repurposing Hub -> PubChem MeSH -> manual/heuristic rules), so Part C can compare SHAP feature importance across MOA groups.

### A.1a Load CTRPv2 compound list

In [ ]:
df_comp = pd.read_parquet(RAW_DIR / "ctrpv2_compounds.parquet")
print(f"CTRPv2 compounds: {len(df_comp)}")
print(f"Columns: {df_comp.columns.tolist()}")
n_with_chembl = df_comp["chembl_id"].notna().sum()
print(f"With ChEMBL ID: {n_with_chembl}/{len(df_comp)} ({n_with_chembl/len(df_comp):.0%})")
df_comp.head(3)


### A.1b Resolve ChEMBL IDs from PubChem

PharmacoDB only provides a ChEMBL ID for ~35% of CTRPv2 compounds, but ~68% have a PubChem CID.

For compounds missing **chembl_id**, we look up ChEMBL via the PubChem cross-reference:

**GET /molecule.json?xref_src=PubChem Compound&xref_id={pubchem_cid}**

Results are cached to **data/raw/chembl_id_pubchem_xref.parquet**.

In [ ]:
CHEMBL_BASE = "https://www.ebi.ac.uk/chembl/api/data"

xref_cache_path = RAW_DIR / "chembl_id_pubchem_xref.parquet"

def resolve_chembl_from_pubchem(pubchem_cid: str, retries: int = 3) -> Optional[str]:
    """Map PubChem CID -> ChEMBL molecule ID via ChEMBL cross-references."""
    if pd.isna(pubchem_cid) or not str(pubchem_cid).strip():
        return None
    url = f"{CHEMBL_BASE}/molecule.json"
    params = {
        "xref_src": "PubChem Compound",
        "xref_id": str(pubchem_cid).strip(),
        "format": "json",
        "limit": 1,
    }
    for attempt in range(retries):
        try:
            r = requests.get(url, params=params, timeout=20)
            r.raise_for_status()
            molecules = r.json().get("molecules", [])
            if molecules:
                return molecules[0].get("molecule_chembl_id")
            return None
        except requests.RequestException:
            time.sleep(1.5 * (attempt + 1))
    return None


n_before = df_comp["chembl_id"].notna().sum()
needs_xref = df_comp["chembl_id"].isna() & df_comp["pubchem_cid"].notna()
print(f"Compounds missing ChEMBL ID but with PubChem CID: {needs_xref.sum()}")

if xref_cache_path.exists():
    df_xref = pd.read_parquet(xref_cache_path)
    print(f"Loaded xref cache: {xref_cache_path.name} ({len(df_xref)} rows)")
else:
    records = []
    todo = df_comp.loc[needs_xref, ["compound", "compound_id", "pubchem_cid"]]
    for _, row in tqdm(todo.iterrows(), total=len(todo), desc="PubChem -> ChEMBL"):
        chembl_id = resolve_chembl_from_pubchem(row["pubchem_cid"])
        records.append({
            "compound": row["compound"],
            "compound_id": row["compound_id"],
            "pubchem_cid": row["pubchem_cid"],
            "chembl_id_resolved": chembl_id,
        })
        time.sleep(0.2)

    df_xref = pd.DataFrame(records)
    df_xref.to_parquet(xref_cache_path, index=False)
    print(f"Saved xref cache: {xref_cache_path.name} ({len(df_xref)} rows)")

n_resolved = df_xref["chembl_id_resolved"].notna().sum()
print(f"PubChem xref hits: {n_resolved}/{len(df_xref)} "
      f"({n_resolved / max(len(df_xref), 1):.0%})")

# Backfill df_comp
xref_map = (
    df_xref.dropna(subset=["chembl_id_resolved"])
    .set_index("compound_id")["chembl_id_resolved"]
    .to_dict()
)
fill_mask = df_comp["chembl_id"].isna() & df_comp["compound_id"].isin(xref_map)
df_comp.loc[fill_mask, "chembl_id"] = df_comp.loc[fill_mask, "compound_id"].map(xref_map)

n_after = df_comp["chembl_id"].notna().sum()
print(f"ChEMBL ID coverage: {n_before}/{len(df_comp)} -> "
      f"{n_after}/{len(df_comp)} (+{n_after - n_before} from PubChem xref)")

moa_cache_path = RAW_DIR / "ctrpv2_moa_chembl.parquet"
if n_after > n_before and moa_cache_path.exists():
    moa_cache_path.unlink()
    print(f"\nInvalidated stale cache: {moa_cache_path.name} "
          f"(deleted automatically -- {n_after - n_before} new chembl_ids "
          f"need their mechanism fetched in A.2)")
elif n_after > n_before:
    print(f"\n{n_after - n_before} new chembl_ids resolved; "
          f"no existing MOA cache to invalidate (first run).")

### A.2 Fetch MOA from ChEMBL API

ChEMBL **/mechanism** endpoint returns the mechanism of action for each molecule.

We query by **molecule_chembl_id** and take the primary mechanism.

Results are cached to **data/raw/ctrpv2_moa_chembl.parquet**

In [ ]:
CHEMBL_BASE = "https://www.ebi.ac.uk/chembl/api/data"

def get_chembl_moa(chembl_id: str, retries: int = 3) -> dict:
    """
    Query ChEMBL for mechanism of action.
    Returns dict with keys: mechanism_of_action, action_type, target_chembl_id.
    Returns {} if not found.
    """
    if pd.isna(chembl_id) or not str(chembl_id).strip():
        return {}
    url = f"{CHEMBL_BASE}/mechanism.json"
    params = {"molecule_chembl_id": str(chembl_id).strip(),
              "format": "json", "limit": 10}
    for attempt in range(retries):
        try:
            r = requests.get(url, params=params, timeout=20)
            r.raise_for_status()
            mechanisms = r.json().get("mechanisms", [])
            if not mechanisms:
                return {}
            m = mechanisms[0]
            return {
                "mechanism_of_action": m.get("mechanism_of_action"),
                "action_type":         m.get("action_type"),
                "target_chembl_id":    m.get("target_chembl_id"),
            }
        except requests.RequestException:
            time.sleep(1.5 * (attempt + 1))
    return {}


In [ ]:
moa_cache_path = RAW_DIR / "ctrpv2_moa_chembl.parquet"

if moa_cache_path.exists():
    df_moa_raw = pd.read_parquet(moa_cache_path)
    print(f"Loaded from cache: {moa_cache_path.name} ({len(df_moa_raw)} rows)")
else:
    records = []
    for _, row in tqdm(df_comp.iterrows(), total=len(df_comp), desc="ChEMBL MOA"):
        result = get_chembl_moa(row.get("chembl_id"))
        records.append({
            "compound":            row["compound"],
            "compound_id":         row["compound_id"],
            "chembl_id":           row.get("chembl_id"),
            "mechanism_of_action": result.get("mechanism_of_action"),
            "action_type":         result.get("action_type"),
            "target_chembl_id":    result.get("target_chembl_id"),
        })
        time.sleep(0.2)

    df_moa_raw = pd.DataFrame(records)
    df_moa_raw.to_parquet(moa_cache_path, index=False)
    print(f"Saved: {moa_cache_path.name} ({len(df_moa_raw)} rows)")

n_annotated = df_moa_raw["mechanism_of_action"].notna().sum()
print(f"Annotated: {n_annotated}/{len(df_moa_raw)} "
      f"({n_annotated/len(df_moa_raw):.0%})")

print("\nTop raw MOA strings (most frequent):")
print(df_moa_raw["mechanism_of_action"].value_counts().head(15).to_string())


### A.3 Map to coarse MOA classes

Raw ChEMBL MOA strings are very specific (e.g. "EGFR inhibitor", "PARP inhibitor").

Сollapse them into ~10 interpretable classes for per-class SHAP comparison.

Compounds without a resolvable annotation -> **"Other/Unknown"**.

In [ ]:
MOA_RULES = [
    # DNA damage
    ("parp",                 "DNA damage/repair"),
    ("topoisomerase",        "DNA damage/repair"),
    ("alkylat",              "DNA damage/repair"),
    ("intercalat",           "DNA damage/repair"),
    ("dna",                  "DNA damage/repair"),
    ("checkpoint kinase",    "DNA damage/repair"),  # CHK1/CHK2
    ("atr ",                 "DNA damage/repair"),
    ("atm ",                 "DNA damage/repair"),

    # Kinase
    ("kinase",               "Kinase inhibitor"),
    ("epidermal growth factor receptor",  "Kinase inhibitor"),  # EGFR
    ("vascular endothelial growth factor","Kinase inhibitor"),  # VEGFR
    ("mtor",                 "Kinase inhibitor"),
    ("cyclin",               "Kinase inhibitor"),   # CDK "cyclin-dependent"
    ("abl",                  "Kinase inhibitor"),
    ("braf",                 "Kinase inhibitor"),
    ("mek",                  "Kinase inhibitor"),
    ("pi3k",                 "Kinase inhibitor"),
    ("phosphoinositide",     "Kinase inhibitor"),   # PI3K
    ("akt",                  "Kinase inhibitor"),
    ("raf",                  "Kinase inhibitor"),
    ("erk",                  "Kinase inhibitor"),
    ("src",                  "Kinase inhibitor"),

    # Cytoskeleton
    ("tubulin",              "Cytoskeleton/cell cycle"),
    ("microtubule",          "Cytoskeleton/cell cycle"),
    ("aurora",               "Cytoskeleton/cell cycle"),
    ("polo",                 "Cytoskeleton/cell cycle"),   # PLK = polo-like kinase
    ("plk",                  "Cytoskeleton/cell cycle"),
    ("kinesin",              "Cytoskeleton/cell cycle"),
    ("spindle",              "Cytoskeleton/cell cycle"),

    # Epigenetic
    ("histone",              "Epigenetic"),
    ("hdac",                 "Epigenetic"),
    ("methyltransfer",       "Epigenetic"),
    ("bromodomain",          "Epigenetic"),
    ("bet ",                 "Epigenetic"),
    ("ezh2",                 "Epigenetic"),

    # Proteasome / protein homeostasis
    ("proteasome",           "Proteasome/protein"),
    ("heat shock",           "Proteasome/protein"),   # HSP90
    ("hsp",                  "Proteasome/protein"),
    ("mdm2",                 "Proteasome/protein"),
    ("ubiquitin",            "Proteasome/protein"),

    # Receptor
    ("receptor",             "Receptor"),
    ("estrogen",             "Receptor"),
    ("androgen",             "Receptor"),
    ("gpcr",                 "Receptor"),

    # Nuclear export / intracellular trafficking
    ("exportin",             "Nuclear export/trafficking"),      # Leptomycin B target
    ("xpo1",                 "Nuclear export/trafficking"),
    ("crm1",                 "Nuclear export/trafficking"),
    ("nuclear import",       "Nuclear export/trafficking"),
    ("importin",             "Nuclear export/trafficking"),
    ("clathrin",             "Nuclear export/trafficking"),
    ("endocytosis",          "Nuclear export/trafficking"),
    ("efflux transporter",   "Nuclear export/trafficking"),      # e.g. BCRP/ABCG2
    ("abcg2",                "Nuclear export/trafficking"),
    ("bcrp",                 "Nuclear export/trafficking"),

    # Metabolic / antimetabolite
    ("atp synthase",         "Metabolic"),            # Oligomycin A
    ("oxidative phosphoryl", "Metabolic"),
    ("dhfr",                 "Metabolic"),
    ("thymidylate",          "Metabolic"),
    ("metabol",              "Metabolic"),
    ("nucleoside",           "Metabolic"),
    ("purine",               "Metabolic"),
    ("pyrimidine",           "Metabolic"),
    ("glutaminase",          "Metabolic"),
    ("glutathione",          "Metabolic"),           # GPX4 (RSL-3)
    ("gpx",                  "Metabolic"),
    ("lipoxygenase",         "Metabolic"),
    ("lipid peroxid",        "Metabolic"),
    ("cyclooxygenase",       "Metabolic"),           # COX inhibitors

    # Apoptosis regulation (BCL-2/IAP family & related cell-death machinery)
    ("bcl-2",                "Apoptosis regulation"),
    ("bcl2",                 "Apoptosis regulation"),
    ("bcl-xl",               "Apoptosis regulation"),
    ("mcl-1",                "Apoptosis regulation"),
    ("survivin",             "Apoptosis regulation"),
    ("birc5",                "Apoptosis regulation"),
    ("caspase",              "Apoptosis regulation"),
    ("iap ",                 "Apoptosis regulation"),            # inhibitor-of-apoptosis family

    # Transcription factor / gene regulation (non-kinase, non-epigenetic)
    ("transcription factor", "Transcription factor/gene regulation"),
    ("stat3",                "Transcription factor/gene regulation"),
    ("stat5",                "Transcription factor/gene regulation"),
    ("p53",                  "Transcription factor/gene regulation"),
    ("nfkb",                 "Transcription factor/gene regulation"),
    ("nf-kb",                "Transcription factor/gene regulation"),
]


In [ ]:
COMPOUND_NAME_RULES = [
    ("vincristine",          "Cytoskeleton/cell cycle"),
    ("docetaxel",            "Cytoskeleton/cell cycle"),
    ("paclitaxel",           "Cytoskeleton/cell cycle"),
    ("cytarabine",           "DNA damage/repair"),
    ("gemcitabine",          "Metabolic"),
    ("oligomycin",           "Metabolic"),
    ("leptomycin",           "Nuclear export/trafficking"),
    ("rsl-3",                "Metabolic"),          # ferroptosis / GPX4
    ("1s,3r-rsl",            "Metabolic"),
    ("birinapant",           "Proteasome/protein"), # IAP inhibitor
    ("navitoclax",           "Proteasome/protein"), # BCL2 inhibitor
    ("ouabain",              "Receptor"),           # Na/K-ATPase

    # Intracellular trafficking
    ("ko-143",               "Nuclear export/trafficking"),  # BCRP/ABCG2 efflux inhibitor
    ("importazole",          "Nuclear export/trafficking"),  # importin-beta / nuclear import
    ("brefeldin",            "Nuclear export/trafficking"),  # ARF1 GEF / ER-to-Golgi
    ("pitstop",              "Nuclear export/trafficking"),  # clathrin-mediated endocytosis

    # Apoptosis regulation (BCL-2/IAP family & related cell-death machinery)
    ("obatoclax",            "Apoptosis regulation"),
    ("tw 37",                "Apoptosis regulation"),        # BCL-2 family inhibitor
    ("gossypol",             "Apoptosis regulation"),
    ("abt-737",              "Apoptosis regulation"),        # BCL-2/BCL-XL inhibitor
    ("venetoclax",           "Apoptosis regulation"),        # BCL-2 selective inhibitor
    ("marinopyrrole",        "Apoptosis regulation"),        # MCL-1 degrader
    ("sepantronium",         "Apoptosis regulation"),        # survivin/BIRC5 inhibitor
    ("bax channel blocker",  "Apoptosis regulation"),
    ("pac-1",                "Apoptosis regulation"),        # procaspase-3 activator
    ("avicin",               "Apoptosis regulation"),        # broad apoptosis induction

    # Transcription factor / gene regulation
    ("yk 4-279",             "Transcription factor/gene regulation"),  # EWS-FLI1 PPI inhibitor
    ("cucurbitacin",         "Transcription factor/gene regulation"),  # STAT3 inhibitor
    ("pifithrin",            "Transcription factor/gene regulation"),  # p53 transcriptional-activity inhibitor
    ("fqi-1",                "Transcription factor/gene regulation"),  # LSF/TFCP2 inhibitor
    ("fqi-2",                "Transcription factor/gene regulation"),  # LSF/TFCP2 inhibitor
    ("prima-1",              "Transcription factor/gene regulation"),  # mutant p53 reactivator
    ("parthenolide",         "Transcription factor/gene regulation"),  # NF-kB inhibitor
]


In [ ]:
def assign_moa_class(moa_str: str, compound_name: str = "") -> str:
    if not pd.isna(moa_str):
        low = moa_str.lower()
        for keyword, cls in MOA_RULES:
            if keyword.lower() in low:
                return cls

    cname = compound_name.lower()
    for keyword, cls in COMPOUND_NAME_RULES:
        if keyword in cname:
            return cls

    return "Other/Unknown"

In [ ]:
# Smoke test - against silent regressions whenever MOA_RULES /
# COMPOUND_NAME_RULES are edited.

_SMOKE_TESTS = [
    ("HISTONE LYSINE METHYLTRANSFERASE INHIBITOR", "BIX-01294",   "Epigenetic"),
    ("KINESIN INHIBITOR",                          "BRD9876",     "Cytoskeleton/cell cycle"),
    ("HDAC INHIBITOR",                             "Panobinostat","Epigenetic"),
    ("Tubulin stabiliser",                         "Docetaxel",   "Cytoskeleton/cell cycle"),
    ("CYCLOOXYGENASE INHIBITOR",                   "Curcumin",    "Metabolic"),
    (None,                                         "Vincristine", "Cytoskeleton/cell cycle"),
    (None,                                         "Venetoclax",  "Apoptosis regulation"),
    (None,                                         "Obatoclax",   "Apoptosis regulation"),
    (None,                                         "pifithrin-alpha", "Transcription factor/gene regulation"),
    ("NF-kB pathway inhibitor",                    "Some Compound", "Transcription factor/gene regulation"),
    (None,                                         "Leptomycin B", "Nuclear export/trafficking"),
    (None,                                         "Ko-143",      "Nuclear export/trafficking"),
]
for moa_str, name, expected in _SMOKE_TESTS:
    got = assign_moa_class(moa_str, name)
    assert got == expected, f"{name}: expected {expected!r}, got {got!r}"
print(f"Smoke test passed: {len(_SMOKE_TESTS)}/{len(_SMOKE_TESTS)} known cases classify correctly.")


### A.4 Baseline classification (ChEMBL-id + manual rules only)

Establishes a reference point before adding the extra data sources below, so the coverage gain from the waterfall (A.6) is measurable.

In [ ]:
df_moa_raw["moa_class"] = df_moa_raw.apply(
    lambda row: assign_moa_class(
        row["mechanism_of_action"],
        row.get("compound", "")
    ),
    axis=1
)

baseline_resolved = (df_moa_raw["moa_class"] != "Other/Unknown").sum()
print(f"Baseline (ChEMBL-id + manual rules only): "
      f"{baseline_resolved}/{len(df_moa_raw)} resolved "
      f"({baseline_resolved/len(df_moa_raw):.0%})")
print("\nMOA class distribution:")
print(df_moa_raw["moa_class"].value_counts().to_string())


### A.5 Additional MOA data sources

Helper functions for the waterfall in A.6: compound-name normalization, ChEMBL lookup by InChIKey, the PRISM Repurposing Hub (via the Figshare API), a PubChem MeSH backstop, and a low-confidence INN/USAN suffix heuristic.

In [ ]:
# Name normalization - needed for matching CTRPv2 names against PRISM names
_SALT_SUFFIXES = (
    "hydrochloride", "hcl", "sodium", "potassium", "sulfate", "sulphate",
    "citcitrate", "citrate", "mesylate", "tartrate", "phosphate", "maleate",
    "acetate", "dihydrate", "hemihydrate", "monohydrate", "besylate",
    "fumarate", "succinate", "hydrobromide", "free base", "freebase",
)
_STEREO_PREFIXES = (
    r"^\(\+/-\)-", r"^\(±\)-", r"^\(\+\)-", r"^\(-\)-",
    r"^\(r\)-", r"^\(s\)-", r"^\(r,s\)-", r"^\(rs\)-",
)

def normalize_compound_name(name: str) -> str:
    if pd.isna(name):
        return ""
    n = str(name).lower().strip()
    for pat in _STEREO_PREFIXES:
        n = re.sub(pat, "", n)

    changed = True
    while changed:
        changed = False
        for salt in _SALT_SUFFIXES:
            new_n = re.sub(rf"\s*{re.escape(salt)}\s*$", "", n)
            if new_n != n:
                n = new_n
                changed = True

    n = re.sub(r"[^a-z0-9]+", "", n)
    return n

In [ ]:
# ChEMBL by InChIKey - recovers molecules ChEMBL knows about even when
# PharmacoDB's own chembl_id column is empty

def get_chembl_id_by_inchikey(inchikey: str, retries: int = 3) -> str | None:
    """Look up a ChEMBL ID by exact standard InChIKey match."""
    if pd.isna(inchikey) or not str(inchikey).strip():
        return None
    url = f"{CHEMBL_BASE}/molecule.json"
    params = {
        "molecule_structures__standard_inchi_key": str(inchikey).strip(),
        "format": "json",
    }
    for attempt in range(retries):
        try:
            r = requests.get(url, params=params, timeout=20)
            r.raise_for_status()
            mols = r.json().get("molecules", [])
            if mols:
                return mols[0]["molecule_chembl_id"]
            return None
        except requests.RequestException:
            time.sleep(1.5 * (attempt + 1))
    return None

def get_chembl_moa_by_inchikey(inchikey: str) -> dict:
    """Full pipeline: InChIKey -> ChEMBL ID -> mechanism_of_action."""
    chembl_id = get_chembl_id_by_inchikey(inchikey)
    if chembl_id is None:
        return {}
    return get_chembl_moa(chembl_id)


In [ ]:
# Broad PRISM Repurposing Hub, queried via the Figshare API
FIGSHARE_API = "https://api.figshare.com/v2"
HEADERS = {"User-Agent": "Mozilla/5.0"}

# Filenames that historically carry the moa/target columns across releases
TREATMENT_FILE_PATTERNS = [
    r"treatment_info",
    r"compound_list",
    r"primary[-_]replicate[-_]treatment",
]

_BRD_ID_RE = re.compile(r"BRD-[A-Z]\d{8}", re.IGNORECASE)

def extract_brd_id(text) -> str | None:
    if pd.isna(text):
        return None
    m = _BRD_ID_RE.search(str(text))
    return m.group(0).upper() if m else None


def _search_prism_articles() -> list[dict]:
    """
    Search Figshare for PRISM Repurposing dataset articles.
    """
    search_terms = ["PRISM Repurposing", "Repurposing Public", "DepMap Repurposing"]
    seen_ids = set()
    articles: list[dict] = []

    for term in search_terms:
        resp = requests.post(
            f"{FIGSHARE_API}/articles/search",
            headers={**HEADERS, "Content-Type": "application/json"},
            json={
                "search_for": term,
                "order": "published_date",
                "order_direction": "desc",
                "page_size": 30,
            },
            timeout=60,
        )
        if not resp.ok:
            print(f"  Figshare search failed for '{term}': HTTP {resp.status_code}")
            continue
        for a in resp.json():
            if a["id"] not in seen_ids:
                seen_ids.add(a["id"])
                articles.append(a)

    articles = [
        a for a in articles
        if re.search(r"repurposing|prism", a.get("title", ""), re.IGNORECASE)
    ]

    print(f"  Figshare search: {len(articles)} candidate PRISM/Repurposing articles found")
    for a in articles[:10]:
        print(f"    - [{a['id']}] {a['title']}")

    return articles

def _extract_quarter(title: str) -> tuple[int, int]:
    """Extract (year, quarter) from a title like 'Repurposing Public 24Q2' for sorting."""
    m = re.search(r"(\d{2})Q(\d)", title)
    if not m:
        return (0, 0)
    return (int(m.group(1)), int(m.group(2)))


def find_prism_treatment_info_url(debug: bool = True) -> tuple[str, str] | None:
    """
    Finds the most recent PRISM Repurposing treatment/compound-info file
    by querying the Figshare API directly
    """
    articles = _search_prism_articles()
    if not articles:
        print("No PRISM/Repurposing articles found on Figshare")
        return None

    articles.sort(key=lambda a: _extract_quarter(a.get("title", "")), reverse=True)

    all_seen_files = []

    for article in articles:
        article_id = article["id"]
        files_resp = requests.get(
            f"{FIGSHARE_API}/articles/{article_id}/files", headers=HEADERS, timeout=60,
        )
        if not files_resp.ok:
            if debug:
                print(f"  [{article_id}] could not list files: HTTP {files_resp.status_code}")
            continue
        files = files_resp.json()

        if debug:
            print(f"  [{article_id}] '{article['title']}' -- {len(files)} files:")
            for f in files:
                print(f"      {f['name']}")
        all_seen_files.extend((article["title"], f["name"], f["download_url"]) for f in files)

        for f in files:
            fname = f["name"]
            if any(re.search(pat, fname, re.IGNORECASE) for pat in TREATMENT_FILE_PATTERNS):
                print(f"Found candidate in '{article['title']}': {fname}")
                return f["download_url"], fname

    loose_matches = [t for t in all_seen_files if re.search(r"info|meta|compound", t[1], re.IGNORECASE)]
    if loose_matches:
        print("No strict match:")
        for title, fname, url in loose_matches:
            print(f"    '{title}': {fname}  ->  {url}")

    print("No treatment/compound-info file found among recent PRISM Repurposing releases.")
    return None


def download_prism_treatment_info(dest: Path) -> bool:
    """Downloads the PRISM treatment-info file if not already cached."""
    if dest.exists():
        print(f"Already downloaded: {dest.name}")
        return True

    result = find_prism_treatment_info_url()
    if result is None:
        print("No PRISM Repurposing treatment-info file found via Figshare "
              "-- skipping this MOA source.")
        return False

    url, fname = result
    print(f"Found PRISM treatment info: {fname}  (downloading from Figshare...)")
    resp = requests.get(url, headers=HEADERS, timeout=300)
    resp.raise_for_status()
    dest.write_bytes(resp.content)
    print(f"  Saved: {dest.name} ({dest.stat().st_size / 1024:.0f} KB)")
    return True


def build_prism_lookup(prism_path: Path) -> dict:
    """
    Loads the PRISM treatment-info CSV and builds TWO lookup tables:
      - "by_name": normalized compound name (+ synonyms) -> (moa, target)
      - "by_id":   Broad ID (extracted from the PRISM "IDs" column) -> (moa, target)

    The by_id table exists specifically to resolve CTRPv2 compounds that
    are literally named by their Broad ID (e.g. "BRD-K63431240") -- these
    never match on drug name/synonym, since the "name" IS an internal ID,
    not a chemical/brand name.
    """
    df_prism = pd.read_csv(prism_path)

    name_col = next((c for c in ["name", "compound_name", "pert_iname", "Drug.Name", "Drug_Name"]
                      if c in df_prism.columns), None)
    moa_col = next((c for c in ["moa", "MOA"] if c in df_prism.columns), None)
    target_col = next((c for c in ["target", "targets", "repurposing_target"]
                        if c in df_prism.columns), None)
    id_col = next((c for c in ["IDs", "ids", "broad_id", "BroadID"]
                   if c in df_prism.columns), None)

    empty = pd.DataFrame(columns=["moa", "target"])
    if name_col is None or moa_col is None:
        print(f"Expected columns not found in PRISM file. "
              f"Available columns: {df_prism.columns.tolist()}")
        return {"by_name": empty, "by_id": empty}

    df_prism = df_prism.dropna(subset=[moa_col]).copy()
    df_prism["name_norm"] = df_prism[name_col].apply(normalize_compound_name)
    df_prism_dedup = df_prism.drop_duplicates("name_norm")

    by_name = df_prism_dedup.set_index("name_norm")[[moa_col]].rename(columns={moa_col: "moa"})
    by_name["target"] = df_prism_dedup.set_index("name_norm")[target_col] if target_col else None

    if "Synonyms" in df_prism.columns:
        syn_rows = []
        for _, row in df_prism.dropna(subset=["Synonyms"]).iterrows():
            for syn in str(row["Synonyms"]).split(","):
                syn_norm = normalize_compound_name(syn)
                if syn_norm:
                    syn_rows.append({
                        "name_norm": syn_norm,
                        "moa": row[moa_col],
                        "target": row.get(target_col) if target_col else None,
                    })
        if syn_rows:
            syn_lookup = pd.DataFrame(syn_rows).drop_duplicates("name_norm").set_index("name_norm")
            by_name = pd.concat([by_name, syn_lookup[~syn_lookup.index.isin(by_name.index)]])

    by_id = empty
    if id_col:
        id_rows = []
        for _, row in df_prism.iterrows():
            brd_id = extract_brd_id(row.get(id_col))
            if brd_id:
                id_rows.append({
                    "brd_id": brd_id,
                    "moa": row[moa_col],
                    "target": row.get(target_col) if target_col else None,
                })
        if id_rows:
            by_id = pd.DataFrame(id_rows).drop_duplicates("brd_id").set_index("brd_id")

    print(f"PRISM lookup built: {len(by_name)} unique compound names/synonyms, "
          f"{len(by_id)} unique Broad IDs with MOA")
    return {"by_name": by_name, "by_id": by_id}


def get_prism_moa(compound_name: str, prism_lookup: dict) -> dict:
    """
    Looks up MOA/target for a compound. Tries a Broad-ID match first (exact
    and unambiguous when the compound name IS a Broad ID), then falls back
    to normalized-name/synonym matching.
    """
    by_id = prism_lookup.get("by_id", pd.DataFrame())
    brd_id = extract_brd_id(compound_name)
    if brd_id and not by_id.empty and brd_id in by_id.index:
        row = by_id.loc[brd_id]
        if isinstance(row, pd.DataFrame):
            row = row.iloc[0]
        return {"mechanism_of_action": row["moa"], "target": row.get("target")}

    by_name = prism_lookup.get("by_name", pd.DataFrame())
    key = normalize_compound_name(compound_name)
    if not by_name.empty and key in by_name.index:
        row = by_name.loc[key]
        if isinstance(row, pd.DataFrame):
            row = row.iloc[0]
        return {"mechanism_of_action": row["moa"], "target": row.get("target")}

    return {}


In [ ]:
# PubChem MeSH Pharmacological Classification - backstop via pubchem_cid
def get_pubchem_pharm_class(cid, retries: int = 2) -> list[str]:
    """
    Returns MeSH Pharmacological Classification terms for a PubChem CID.
    Returns [] if the CID has no such annotation.
    """
    if pd.isna(cid):
        return []
    try:
        cid_int = int(float(cid))
    except (ValueError, TypeError):
        return []

    url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug_view/data/compound/{cid_int}/JSON"
    for attempt in range(retries):
        try:
            r = requests.get(url, params={"heading": "MeSH Pharmacological Classification"}, timeout=20)
            if r.status_code == 404:
                return []
            r.raise_for_status()
            data = r.json()
            terms = []

            def walk(sections):
                for sec in sections:
                    if sec.get("TOCHeading") == "MeSH Pharmacological Classification":
                        for info in sec.get("Information", []):
                            for markup in info.get("Value", {}).get("StringWithMarkup", []):
                                terms.append(markup["String"])
                    if "Section" in sec:
                        walk(sec["Section"])

            walk(data.get("Record", {}).get("Section", []))
            return terms
        except requests.RequestException:
            time.sleep(1.0 * (attempt + 1))
    return []


def fetch_all_pubchem_pharm_classes(cids: pd.Series, cache_path: Path) -> dict:
    """
    Fetches MeSH pharm class terms for a list of PubChem CIDs, with disk
    caching.
    """
    if cache_path.exists():
        cache_df = pd.read_parquet(cache_path)
        return dict(zip(cache_df["pubchem_cid"], cache_df["mesh_terms"]))

    from tqdm.auto import tqdm

    results = {}
    for cid in tqdm(cids.dropna().unique(), desc="PubChem MeSH"):
        terms = get_pubchem_pharm_class(cid)
        results[cid] = terms
        time.sleep(0.25)

    cache_df = pd.DataFrame({
        "pubchem_cid": list(results.keys()),
        "mesh_terms": [";".join(v) for v in results.values()],
    })
    cache_df.to_parquet(cache_path, index=False)
    return {k: v for k, v in results.items()}


In [ ]:
# INN/USAN suffix heuristic
INN_STEM_RULES = [
    ("tinib",    "Kinase inhibitor"),          # imatinib, erlotinib, osimertinib
    ("ciclib",   "Kinase inhibitor"),          # palbociclib, ribociclib (CDK4/6)
    ("taxel",    "Cytoskeleton/cell cycle"),   # paclitaxel, docetaxel
    ("platin",   "DNA damage/repair"),         # cisplatin, carboplatin, oxaliplatin
    ("rubicin",  "DNA damage/repair"),         # doxorubicin, epirubicin (anthracyclines)
    ("mustine",  "DNA damage/repair"),         # chlormethine-type alkylators
    ("degib",    "Receptor"),                  # vismodegib, sonidegib (hedgehog pathway)
    ("vastatin", "Metabolic"),                 # atorvastatin, simvastatin (HMG-CoA reductase)
    ("nib",      "Kinase inhibitor"),          # broad catch-all, kept last & specific
]

def guess_moa_by_suffix(compound_name: str) -> str | None:
    name_low = str(compound_name).lower()
    for suffix, cls in INN_STEM_RULES:
        if name_low.endswith(suffix):
            return cls
    return None


### A.6 Multi-source MOA waterfall

Combines all sources above into a single priority-ordered pipeline: ChEMBL (by id, then by InChIKey) -> PRISM Repurposing Hub -> PubChem MeSH -> manual name rules -> INN suffix heuristic. First classification hit wins.

In [ ]:
def annotate_compound_moa_extended(
    row: pd.Series,
    prism_lookup: pd.DataFrame,
    pubchem_mesh: dict,
) -> dict:
    """
    Multi-source MOA waterfall for a single compound row.
    Returns dict with keys: moa_class, moa_source, mechanism_raw.
    Tries sources in priority order; first classification hit wins.
    """
    compound = row["compound"]

    # 1. ChEMBL by chembl_id (already stored in df_moa_raw["mechanism_of_action"])
    raw = row.get("mechanism_of_action")
    if pd.notna(raw):
        cls = assign_moa_class(raw, compound)
        if cls != "Other/Unknown":
            return {"moa_class": cls, "moa_source": "chembl_id", "mechanism_raw": raw}

    # 2. ChEMBL by InChIKey
    result = get_chembl_moa_by_inchikey(row.get("inchikey"))
    if result.get("mechanism_of_action"):
        cls = assign_moa_class(result["mechanism_of_action"], compound)
        if cls != "Other/Unknown":
            return {"moa_class": cls, "moa_source": "chembl_inchikey",
                    "mechanism_raw": result["mechanism_of_action"]}

    # 3. PRISM Repurposing Hub, by Broad ID first, then by name/synonyms
    result = get_prism_moa(compound, prism_lookup)
    if result.get("mechanism_of_action"):
        cls = assign_moa_class(result["mechanism_of_action"], compound)
        if cls != "Other/Unknown":
            return {"moa_class": cls, "moa_source": "prism_repurposing",
                    "mechanism_raw": result["mechanism_of_action"]}

    # 4. PubChem MeSH Pharmacological Classification
    terms_str = pubchem_mesh.get(row.get("pubchem_cid"), "")
    if terms_str:
        cls = assign_moa_class(terms_str.replace(";", " "), compound)
        if cls != "Other/Unknown":
            return {"moa_class": cls, "moa_source": "pubchem_mesh", "mechanism_raw": terms_str}

    # 5. Manual compound-name rules
    cname = str(compound).lower()
    for keyword, cls in COMPOUND_NAME_RULES:
        if keyword in cname:
            return {"moa_class": cls, "moa_source": "manual_name_rule", "mechanism_raw": None}

    # 6. INN/USAN suffix heuristic
    cls = guess_moa_by_suffix(compound)
    if cls:
        return {"moa_class": cls, "moa_source": "inn_suffix_heuristic", "mechanism_raw": None}

    return {"moa_class": "Other/Unknown", "moa_source": "none", "mechanism_raw": None}


In [ ]:
# PRISM Repurposing Hub
prism_path = RAW_DIR / "prism_repurposing_treatment_info.csv"
prism_ok = download_prism_treatment_info(prism_path)
prism_lookup = build_prism_lookup(prism_path) if prism_ok else {
    "by_name": pd.DataFrame(columns=["moa", "target"]),
    "by_id":   pd.DataFrame(columns=["moa", "target"]),
}


In [ ]:
# PubChem MeSH terms (cached, one entry per unique CID)
pubchem_cache_path = RAW_DIR / "pubchem_mesh_pharm_class.parquet"
pubchem_mesh = fetch_all_pubchem_pharm_classes(df_comp["pubchem_cid"], pubchem_cache_path)


In [ ]:
# Run the waterfall over all compounds
ext_results = df_moa_raw.apply(
    lambda row: annotate_compound_moa_extended(row, prism_lookup, pubchem_mesh),
    axis=1,
    result_type="expand",
)
df_moa_raw[["moa_class", "moa_source", "mechanism_raw_extended"]] = ext_results


### A.7 Coverage & diagnostics

In [ ]:
# Coverage report: how many compounds each source resolved
print("MOA resolution waterfall:")
source_counts = df_moa_raw["moa_source"].value_counts()
print(source_counts.to_string())

n_resolved = (df_moa_raw["moa_class"] != "Other/Unknown").sum()
print(f"\nTotal resolved: {n_resolved}/{len(df_moa_raw)} "
      f"({n_resolved/len(df_moa_raw):.0%})  (baseline was {baseline_resolved}/{len(df_moa_raw)})")

print("\nFinal MOA class distribution:")
print(df_moa_raw["moa_class"].value_counts().to_string())


In [ ]:
# Diagnostic
unresolved = df_moa_raw[df_moa_raw["moa_class"] == "Other/Unknown"].copy()
has_raw_text = unresolved["mechanism_raw_extended"].notna() | unresolved["mechanism_of_action"].notna()

print(f"Unresolved: {len(unresolved)}")
print(f"  ...but HAD raw mechanism text (rules just didn't catch it): {has_raw_text.sum()}")
print(f"  ...found NOTHING at all in any source:                     {(~has_raw_text).sum()}")

if has_raw_text.sum():
    print("\nSample raw texts that slipped through the rules (candidates for new MOA_RULES keywords):")
    print(unresolved.loc[has_raw_text, ["compound", "mechanism_raw_extended"]].head(20).to_string(index=False))


In [ ]:
# Sanity check: inspect a sample of the riskiest source
suffix_hits = df_moa_raw[df_moa_raw["moa_source"] == "inn_suffix_heuristic"]
if not suffix_hits.empty:
    print(f"{len(suffix_hits)} compounds resolved ONLY by the INN suffix:")
    print(suffix_hits[["compound", "moa_class"]].to_string(index=False))


### A.8 Manual review of remaining unresolved compounds

In [ ]:
still_unknown = df_moa_raw[df_moa_raw["moa_class"] == "Other/Unknown"].copy()
print(f"Still unresolved: {len(still_unknown)} compounds")

has_raw_signal = still_unknown[still_unknown["mechanism_raw_extended"].notna()
                                & (still_unknown["mechanism_raw_extended"] != "")]
print(f"  ...of which {len(has_raw_signal)} have a raw PRISM/PubChem string "
      f"that just didn't match any MOA_RULES keyword")

raw_signal_counts = has_raw_signal["mechanism_raw_extended"].value_counts()
print("\nMost frequent unmatched raw strings (candidates for new MOA_RULES keywords):")
print(raw_signal_counts.head(30).to_string())

near_miss_path = SHAP_DIR / "moa_near_miss_review.csv"
has_raw_signal[["compound", "mechanism_raw_extended", "moa_source"]].sort_values(
    "mechanism_raw_extended"
).to_csv(near_miss_path, index=False)
print(f"\nSaved for review: {near_miss_path}")


In [ ]:
def reclassify_from_cache(row):
    if row["moa_class"] != "Other/Unknown":
        return row["moa_class"], row["moa_source"]

    raw = row.get("mechanism_raw_extended")
    if pd.notna(raw) and raw:
        cls = assign_moa_class(raw, row["compound"])
        if cls != "Other/Unknown":
            return cls, row["moa_source"] + "+expanded_rules"

    return "Other/Unknown", "none"

df_moa_raw[["moa_class", "moa_source"]] = df_moa_raw.apply(
    reclassify_from_cache, axis=1, result_type="expand"
)

n_resolved_after = (df_moa_raw["moa_class"] != "Other/Unknown").sum()
print(f"After expanding MOA_RULES: {n_resolved_after}/{len(df_moa_raw)} resolved")


In [ ]:
eligible_path = RESULTS_DIR / "eligible_compounds.csv"
eligible = pd.read_csv(eligible_path)[["compound", "coverage", "aac_std"]]

remaining = df_moa_raw[df_moa_raw["moa_class"] == "Other/Unknown"].copy()
remaining = remaining.merge(eligible, on="compound", how="inner")
remaining = remaining.sort_values("coverage", ascending=False)

def make_search_links(row):
    import urllib.parse
    q = urllib.parse.quote(str(row["compound"]) + " mechanism of action")
    pubchem_url = f"https://pubchem.ncbi.nlm.nih.gov/#query={urllib.parse.quote(str(row['compound']))}"
    chembl_url  = f"https://www.ebi.ac.uk/chembl/g/#search_results/all/query={urllib.parse.quote(str(row['compound']))}"
    google_url  = f"https://www.google.com/search?q={q}"
    return pd.Series({
        "pubchem_link": pubchem_url,
        "chembl_link":  chembl_url,
        "google_link":  google_url,
    })

remaining = pd.concat([remaining, remaining.apply(make_search_links, axis=1)], axis=1)

ALLOWED_CLASSES = sorted(
    df_moa_raw.loc[df_moa_raw["moa_class"] != "Other/Unknown", "moa_class"].unique().tolist()
) + ["Other/Unknown"]
remaining["manual_moa_class"] = ""

manual_review_path = SHAP_DIR / "moa_manual_labeling_master.csv"
remaining[["compound", "coverage", "aac_std", "manual_moa_class",
           "pubchem_link", "chembl_link", "google_link"]].to_csv(
    manual_review_path, index=False
)
print(f"\nManual-labeling master file saved: {manual_review_path}")
print(f"  {len(remaining)} compounds to label (modeled subset, sorted by coverage)")
print(f"  Allowed values for manual_moa_class column: {ALLOWED_CLASSES}")


In [ ]:
def merge_manual_labels(labeled_path: Path) -> pd.DataFrame:
    """
    Merges a hand-completed copy of the manual-labeling master file back
    into df_moa_raw.
    """
    if not labeled_path.exists():
        print(f"{labeled_path} not found. No manual labels merged.")
        return df_moa_raw

    manual_df = pd.read_csv(labeled_path)
    manual_df = manual_df[manual_df["manual_moa_class"].notna()
                          & (manual_df["manual_moa_class"].str.strip() != "")]

    bad = manual_df[~manual_df["manual_moa_class"].isin(ALLOWED_CLASSES)]
    if not bad.empty:
        print(f"{len(bad)} entries have a manual_moa_class value "
              f"outside the allowed set - these will be ignored:")
        print(bad[["compound", "manual_moa_class"]].to_string(index=False))
        manual_df = manual_df[manual_df["manual_moa_class"].isin(ALLOWED_CLASSES)]

    print(f"Merging {len(manual_df)} manually-labeled compounds")

    label_map = dict(zip(manual_df["compound"], manual_df["manual_moa_class"]))
    mask = df_moa_raw["compound"].isin(label_map)
    df_moa_raw.loc[mask, "moa_class"] = df_moa_raw.loc[mask, "compound"].map(label_map)
    df_moa_raw.loc[mask, "moa_source"] = "manual_team_review"

    return df_moa_raw

In [ ]:
manual_labeled_path = SHAP_DIR / "moa_manual_labeled.csv"
df_moa_raw = merge_manual_labels(manual_labeled_path)

print("\nMOA distribution after manual review:")
print(df_moa_raw["moa_class"].value_counts().to_string())


### A.9 Finalize classes & save

In [ ]:
# Merge tiny classes (< MIN_CLASS_SIZE compounds) into Other/Unknown.
MIN_CLASS_SIZE = 5
class_counts = df_moa_raw["moa_class"].value_counts()
small = class_counts[class_counts < MIN_CLASS_SIZE].index.tolist()
if small:
    print(f"Merging small classes into 'Other/Unknown': {small}")
    df_moa_raw.loc[df_moa_raw["moa_class"].isin(small), "moa_class"] = "Other/Unknown"

df_moa = df_moa_raw[["compound", "compound_id", "chembl_id",
                      "mechanism_of_action", "action_type", "moa_class",
                      "moa_source", "mechanism_raw_extended"]].copy()
moa_save_path = SHAP_DIR / "compound_moa.parquet"
df_moa.to_parquet(moa_save_path, index=False)
print(f"Saved: {moa_save_path.name} ({len(df_moa)} compounds)")


In [ ]:
# Build lookup dict used throughout the rest of the notebook
moa_map = df_moa.set_index("compound")["moa_class"].to_dict()


In [ ]:
eligible_path = RESULTS_DIR / "eligible_compounds.csv"
if eligible_path.exists():
    eligible_names = pd.read_csv(eligible_path)["compound"].tolist()
    df_moa_modeled = df_moa[df_moa["compound"].isin(eligible_names)].copy()
else:
    df_moa_modeled = df_moa.copy()

print(f"MOA distribution - only modeled compounds "
      f"(n={len(df_moa_modeled)} from {len(df_moa)} overall):")
print(df_moa_modeled["moa_class"].value_counts().to_string())


### A.10 Visualize MOA distribution

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
class_counts_final = df_moa["moa_class"].value_counts()
class_counts_final.sort_values().plot(kind="barh", ax=ax, color="steelblue")
ax.set_xlabel("Number of compounds")
ax.set_title("CTRPv2 compounds per MOA class")
for i, v in enumerate(class_counts_final.sort_values()):
    ax.text(v + 0.3, i, str(v), va="center", fontsize=9)
plt.tight_layout()
plt.savefig(SHAP_DIR / "moa_class_distribution.png", dpi=150)
plt.show()

print(f"\nMOA classes used in analysis: "
      f"{sorted(class_counts_final[class_counts_final >= MIN_CLASS_SIZE].index.tolist())}")


## Part B - SHAP Analysis

### B.1 Reconstruct feature matrices

We rebuild the **exact same feature sets** as notebook 02 so that feature
names match what the saved models were trained on.

Key parameters that must match notebook 02:
- **FEATURE_SELECTION_K = 1000** (quick mode) or **None** (full)
- Variance-based selection (same order of features)

In [ ]:
# Load omics layers (already z-scored from data prep)
LAYER_FILE_NAMES = {
    "expr": "expression", "mut": "mutations",
    "cnv": "cnv", "meth": "methylation", "prot": "proteomics",
}

In [ ]:
omics = {}
for key in ACTIVE_LAYERS:
    fname = LAYER_FILE_NAMES[key]
    p = PROCESSED_DIR / f"{fname}.parquet"
    df = pd.read_parquet(p)
    if "ModelID" in df.columns:
        df = df.set_index("ModelID")
    df = df.drop(columns=[c for c in df.columns if str(c).startswith("Unnamed")],
                 errors="ignore")
    omics[key] = df
    print(f"  {key:<5}: {df.shape}")

drug_response = pd.read_parquet(PROCESSED_DIR / "drug_response.parquet")
if "ModelID" in drug_response.columns:
    drug_response = drug_response.set_index("ModelID")

In [ ]:
# Common cells (same intersection as notebook 02)
common_ids = sorted(
    set.intersection(*(set(df.index) for df in omics.values()),
                     set(drug_response.index))
)
omics = {k: df.loc[common_ids].sort_index().fillna(0.0).astype(np.float32)
         for k, df in omics.items()}
drug_response = drug_response.loc[common_ids].sort_index().astype(np.float32)
print(f"\nCommon cell lines: {len(common_ids)}")

In [ ]:
# Variance-based feature selection
def select_features_by_variance(df: pd.DataFrame,
                                 k: Optional[int] = None) -> pd.DataFrame:
    if k is None or k >= df.shape[1]:
        return df.copy()
    variances = df.var(axis=0, ddof=1).fillna(0.0)
    return df.loc[:, variances.sort_values(ascending=False).head(k).index].copy()

# quick mode (RUN_FULL_MODELING=False) -> 1000; full -> None
FEATURE_SELECTION_K = 1000

omics_for_modeling = {k: select_features_by_variance(df, FEATURE_SELECTION_K)
                      for k, df in omics.items()}

print("Feature sets for SHAP (must match notebook 02):")
for k, df in omics_for_modeling.items():
    print(f"  {k:<5}: {df.shape}")

In [ ]:
# Build feature_sets dict
feature_sets: dict[str, pd.DataFrame] = {}

# Single-omics sets (no prefix, clean gene names)
for layer in ACTIVE_LAYERS:
    feature_sets[layer] = omics_for_modeling[layer]

# Early fusion (prefix: "layer::gene")
early_fusion_parts = [
    omics_for_modeling[n].rename(columns=lambda c, n=n: f"{n}::{c}")
    for n in ACTIVE_LAYERS
]
feature_sets["early_fusion"] = pd.concat(early_fusion_parts, axis=1)

print("\nFeature sets available for SHAP:")
for name, df in feature_sets.items():
    print(f"  {name:<16}: {df.shape[1]:>6} features")

# For "early_fusion" mode the feature-to-layer mapping is derived from prefix
def feature_to_layer(feature_name: str,
                     layer_in_single_mode: Optional[str] = None) -> str:
    """Return the omics layer name for a given feature."""
    if "::" in str(feature_name):
        return str(feature_name).split("::", 1)[0]
    return layer_in_single_mode or "unknown"

### B.2 Model loading helpers

In [ ]:
def safe_compound_name(compound: str) -> str:
    """Reproduce notebook 02 _model_save_path sanitization."""
    return (compound.replace(" ", "_")
                    .replace("/", "-")
                    .replace("(", "")
                    .replace(")", ""))

In [ ]:
def load_fold_model(model_family: str,
                    compound: str,
                    strategy: str,
                    fold_id: int) -> Optional[object]:
    """Load a single saved fold-model. Returns None if not found."""
    safe = safe_compound_name(compound)
    path = MODEL_DIR / f"{model_family}__{safe}__{strategy}__fold{fold_id}.pkl"
    if not path.exists():
        return None
    return joblib.load(path)

In [ ]:
def list_saved_models(model_family: str,
                      strategy: str) -> list[str]:
    """List compounds that have at least one saved model for this config."""
    pattern = f"{model_family}__*__{strategy}__fold1.pkl"
    paths = list(MODEL_DIR.glob(pattern))
    compounds = []
    for p in paths:
        safe = p.stem.split(f"{model_family}__")[1].split(f"__{strategy}")[0]
        compounds.append(safe)
    return compounds

In [ ]:
available = list_saved_models(BEST_MODEL_FAMILY, BEST_STRATEGY)
print(f"Saved models found for ({BEST_MODEL_FAMILY}, {BEST_STRATEGY}): "
      f"{len(available)} compounds")
if not available:
    print("\nWARNING: No models found. Check that notebook 02 ran with "
          "TOP_CONFIGS=None (save all) or that the best config was in TOP_CONFIGS.")

### B.3 SHAP computation helpers

Selects the right explainer automatically based on model type.

In [ ]:
def _get_shap_values_linear(model, X_train_bg: np.ndarray,
                             X_explain: np.ndarray,
                             n_features_in: int) -> np.ndarray:
    """
    Compute SHAP values for linear sklearn Pipeline
    (RidgeCV, ElasticNetCV, RidgeSelectK).

    For RidgeSelectK: SelectKBest reduces the feature space.
    We apply the pre-processing steps manually, compute SHAP in the
    reduced space, then expand back to the full feature space using
    the SelectKBest mask.
    """
    # Check if pipeline has a feature selector
    has_selector = (hasattr(model, "named_steps")
                    and "selectkbest" in model.named_steps)

    if has_selector:
        # Transform data through pre-processing steps (all except final estimator)
        transform_steps = model[:-1]
        X_bg_t  = transform_steps.transform(X_train_bg)
        X_exp_t = transform_steps.transform(X_explain)
        estimator = model[-1]   # fitted RidgeCV
    else:
        # No selector - data passes straight to the estimator
        if hasattr(model, "named_steps"):
            estimator = model.named_steps[list(model.named_steps)[-1]]
            # Apply any scaler if present
            if len(model.named_steps) > 1:
                X_bg_t  = model[:-1].transform(X_train_bg)
                X_exp_t = model[:-1].transform(X_explain)
            else:
                X_bg_t, X_exp_t = X_train_bg, X_explain
        else:
            estimator = model
            X_bg_t, X_exp_t = X_train_bg, X_explain

    masker = shap.maskers.Independent(X_bg_t,
                                      max_samples=min(N_BACKGROUND_SAMPLES,
                                                      len(X_bg_t)))
    explainer = shap.LinearExplainer(estimator, masker)
    shap_reduced = explainer.shap_values(X_exp_t)   # (n_samples, n_selected)

    if not has_selector:
        return shap_reduced   # already full-size

    # Expand back: SelectKBest mask -> full feature space
    mask = model.named_steps["selectkbest"].get_support()
    shap_full = np.zeros((shap_reduced.shape[0], n_features_in), dtype=float)
    shap_full[:, mask] = shap_reduced
    return shap_full

In [ ]:
def compute_shap_values(model,
                        X_train: np.ndarray,
                        X_explain: np.ndarray,
                        n_features_in: int) -> np.ndarray:
    """
    Universal SHAP dispatcher.
    Returns np.ndarray of shape (n_samples, n_features_in).
    """
    # Identify the final estimator type
    if hasattr(model, "named_steps"):
        estimator = list(model.named_steps.values())[-1]
    else:
        estimator = model
    etype = type(estimator).__name__

    if etype in {"RidgeCV", "Ridge", "ElasticNetCV", "ElasticNet",
                 "Lasso", "LassoCV", "LinearRegression"}:
        return _get_shap_values_linear(model, X_train, X_explain, n_features_in)

    elif etype in {"XGBRegressor", "LGBMRegressor",
                   "RandomForestRegressor", "GradientBoostingRegressor"}:
        explainer = shap.TreeExplainer(estimator)
        sv = explainer.shap_values(X_explain)
        return np.array(sv)

    else:
        warnings.warn(f"Unknown estimator type '{etype}'. Using shap.Explainer (slow).")
        bg_df = pd.DataFrame(X_train[:N_BACKGROUND_SAMPLES])
        explainer = shap.Explainer(model, bg_df)
        return explainer(pd.DataFrame(X_explain)).values

### B.4 Main SHAP loop

For each compound × fold model: compute SHAP, average across folds.

In [ ]:
shap_cache_genes  = SHAP_DIR / "shap_per_gene_per_compound.parquet"
shap_cache_layers = SHAP_DIR / "shap_per_layer_per_compound.parquet"

if shap_cache_genes.exists() and shap_cache_layers.exists():
    df_shap_genes  = pd.read_parquet(shap_cache_genes)
    df_shap_layers = pd.read_parquet(shap_cache_layers)
    print(f"Loaded from cache:")
    print(f"  {shap_cache_genes.name}:  {df_shap_genes.shape}")
    print(f"  {shap_cache_layers.name}: {df_shap_layers.shape}")
else:
    # Determine feature set for SHAP
    if SHAP_MODE == "early_fusion":
        shap_feature_key = "early_fusion"
        shap_strategy    = "early_fusion"
    else:
        # single_omics: we compute separately per layer, then combine
        shap_feature_key = BEST_FEATURE_KEY   # e.g. "expr"
        shap_strategy    = "single_omics"

    # List compounds with saved models
    eligible_path = RESULTS_DIR / "eligible_compounds.csv"
    if eligible_path.exists():
        modeling_compounds = pd.read_csv(eligible_path)["compound"].tolist()
    else:
        modeling_compounds = list_saved_models(BEST_MODEL_FAMILY, shap_strategy)
    if MAX_COMPOUNDS:
        modeling_compounds = modeling_compounds[:MAX_COMPOUNDS]

    print(f"Computing SHAP for {len(modeling_compounds)} compounds "
          f"(mode={SHAP_MODE}, model={BEST_MODEL_FAMILY}, strategy={shap_strategy})")

    gene_records  = []   # mean |SHAP| per gene per compound
    layer_records = []   # normalized layer importance per compound
    failed        = []

    for compound in tqdm(modeling_compounds, desc="SHAP"):
        # y for this compound - needed to reconstruct fold splits
        if compound not in drug_response.columns:
            failed.append((compound, "not in drug_response"))
            continue
        y = drug_response[compound].dropna()
        valid_ids = y.index.to_numpy()
        if len(valid_ids) < 30:
            failed.append((compound, f"too few samples ({len(valid_ids)})"))
            continue

        if SHAP_MODE == "single_omics":
            # Compute SHAP per layer, then aggregate across layers
            layer_shap_sums = {}   # layer -> summed mean |SHAP|
            all_gene_shap   = {}   # gene -> mean |SHAP| from its layer's model

            for layer in ACTIVE_LAYERS:
                X_layer = feature_sets[layer]
                feature_names = X_layer.columns.to_numpy()
                n_feats = len(feature_names)

                # Accumulate SHAP across folds
                fold_shap_list = []
                for fold_id in range(1, N_SPLITS + 1):
                    model = load_fold_model(BEST_MODEL_FAMILY, compound,
                                            "single_omics", fold_id)
                    if model is None:
                        continue

                    # Reconstruct this fold's train/test split
                    from sklearn.model_selection import KFold
                    kf = KFold(n_splits=N_SPLITS, shuffle=True,
                               random_state=42)
                    splits = list(kf.split(valid_ids))
                    train_idx, test_idx = splits[fold_id - 1]
                    train_ids = valid_ids[train_idx]

                    X_tr = X_layer.loc[train_ids].to_numpy(dtype=np.float32)
                    X_all = X_layer.loc[valid_ids].to_numpy(dtype=np.float32)

                    try:
                        sv = compute_shap_values(model, X_tr, X_all, n_feats)
                        fold_shap_list.append(np.abs(sv).mean(axis=0))
                    except Exception as e:
                        warnings.warn(f"  {compound} {layer} fold{fold_id}: {e}")

                if not fold_shap_list:
                    continue

                mean_abs_shap = np.stack(fold_shap_list).mean(axis=0)

                # Store per-gene
                for gene, val in zip(feature_names, mean_abs_shap):
                    all_gene_shap[f"{layer}::{gene}"] = float(val)

                layer_shap_sums[layer] = float(mean_abs_shap.sum())

            if not layer_shap_sums:
                failed.append((compound, "no SHAP computed"))
                continue

            # Normalize layer importance to sum to 1
            total = sum(layer_shap_sums.values())
            layer_importance = {k: v / total if total > 0 else np.nan
                                for k, v in layer_shap_sums.items()}

        else:  # early_fusion
            X_ef = feature_sets["early_fusion"]
            feature_names = X_ef.columns.to_numpy()
            n_feats = len(feature_names)

            fold_shap_list = []
            for fold_id in range(1, N_SPLITS + 1):
                model = load_fold_model(BEST_MODEL_FAMILY, compound,
                                        "early_fusion", fold_id)
                if model is None:
                    continue

                from sklearn.model_selection import KFold
                kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
                splits = list(kf.split(valid_ids))
                train_idx, _ = splits[fold_id - 1]
                train_ids = valid_ids[train_idx]

                X_tr = X_ef.loc[train_ids].to_numpy(dtype=np.float32)
                X_all = X_ef.loc[valid_ids].to_numpy(dtype=np.float32)

                try:
                    sv = compute_shap_values(model, X_tr, X_all, n_feats)
                    fold_shap_list.append(np.abs(sv).mean(axis=0))
                except Exception as e:
                    warnings.warn(f"  {compound} ef fold{fold_id}: {e}")

            if not fold_shap_list:
                failed.append((compound, "no SHAP computed"))
                continue

            mean_abs_shap = np.stack(fold_shap_list).mean(axis=0)

            # Per-gene (keep full prefixed name)
            all_gene_shap = {fn: float(v)
                             for fn, v in zip(feature_names, mean_abs_shap)}

            # Per-layer: sum |SHAP| for all features belonging to that layer
            layer_shap_sums = {}
            for layer in ACTIVE_LAYERS:
                mask = np.array([feature_to_layer(fn) == layer
                                 for fn in feature_names])
                layer_shap_sums[layer] = float(mean_abs_shap[mask].sum())

            total = sum(layer_shap_sums.values())
            layer_importance = {k: v / total if total > 0 else np.nan
                                for k, v in layer_shap_sums.items()}

        gene_records.append({
            "compound":  compound,
            "moa_class": moa_map.get(compound, "Other/Unknown"),
            **all_gene_shap,
        })
        layer_records.append({
            "compound":  compound,
            "moa_class": moa_map.get(compound, "Other/Unknown"),
            **layer_importance,
        })

    if failed:
        print(f"\nFailed: {len(failed)}")
        for c, reason in failed[:10]:
            print(f"  {c}: {reason}")

    df_shap_genes  = pd.DataFrame(gene_records).set_index("compound")
    df_shap_layers = pd.DataFrame(layer_records).set_index("compound")

    df_shap_genes.to_parquet(shap_cache_genes)
    df_shap_layers.to_parquet(shap_cache_layers)
    print(f"\nSaved:")
    print(f"  {shap_cache_genes.name}:  {df_shap_genes.shape}")
    print(f"  {shap_cache_layers.name}: {df_shap_layers.shape}")

In [ ]:
print(f"\nLayer importance summary (mean across all compounds):")
layer_cols = [c for c in df_shap_layers.columns if c != "moa_class"]
print(df_shap_layers[layer_cols].mean().round(3).to_string())

In [ ]:
class_sizes_shap = df_shap_layers.groupby("moa_class").size()
low_power_classes = class_sizes_shap[class_sizes_shap < 3].index.tolist()
if low_power_classes:
    print(f"Classes with < 3 connections: "
          f"{low_power_classes}")

## Part C - MOA-stratified Analysis

### C.1 Layer importance per MOA class

In [ ]:
# Mean normalised layer importance per MOA class
moa_layer_df = (
    df_shap_layers.groupby("moa_class")[layer_cols]
    .mean()
    .round(3)
)
print("Mean normalised layer importance per MOA class:")
display(moa_layer_df)

moa_layer_df.to_csv(SHAP_DIR / "moa_layer_importance.csv")

### C.2 Heatmap - omics layer importance x MOA class

In [ ]:
# Number of compounds per class (for annotation)
class_sizes = df_shap_layers.groupby("moa_class").size().rename("n_compounds")

fig, ax = plt.subplots(figsize=(max(5, len(layer_cols) * 1.4),
                                 max(4, len(moa_layer_df) * 0.55)))
im = sns.heatmap(
    moa_layer_df,
    annot=True, fmt=".2f",
    cmap="YlOrRd",
    linewidths=0.5,
    ax=ax,
    vmin=0,
)
# Add sample sizes to y-axis labels
new_labels = [
    f"{idx}  (n={class_sizes.get(idx, '?')})"
    for idx in moa_layer_df.index
]
ax.set_yticklabels(new_labels, rotation=0)
ax.set_xlabel("Omics layer", fontsize=12)
ax.set_ylabel("MOA class", fontsize=12)
ax.set_title("Normalised omics layer importance per MOA class\n"
             "(mean |SHAP| fraction, averaged across compounds)", fontsize=12)
plt.tight_layout()
plt.savefig(SHAP_DIR / "moa_layer_importance_heatmap.png", dpi=150)
plt.show()

In [ ]:
# Bar chart version (stacked)
fig, ax = plt.subplots(figsize=(max(6, len(moa_layer_df) * 0.8), 5))
moa_layer_df.plot(
    kind="bar", stacked=True, ax=ax,
    colormap="tab10", width=0.7,
)
ax.set_xlabel("MOA class")
ax.set_ylabel("Mean normalised |SHAP| fraction")
ax.set_title("Omics layer contribution per drug MOA class")
ax.legend(title="Layer", bbox_to_anchor=(1.01, 1), loc="upper left")
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(SHAP_DIR / "moa_layer_importance_barplot.png", dpi=150)
plt.show()

### C.3 Top genes per MOA class

For each MOA class, find the top-N genes by mean |SHAP| across all compounds in that class. These gene lists are the direct input to KEGG/GO analysis.

In [ ]:
gene_cols = [c for c in df_shap_genes.columns if c != "moa_class"]

def top_genes_for_class(moa_class: str,
                         df_genes: pd.DataFrame,
                         df_layers: pd.DataFrame,
                         top_n: int = TOP_N_GENES_PER_MOA) -> pd.DataFrame:
    """
    For a MOA class, return top-N genes by mean |SHAP|, with layer annotation.
    Gene names in single_omics mode are prefixed: "expr::TP53".
    We strip the prefix for the export.
    """
    compounds_in_class = df_layers[df_layers["moa_class"] == moa_class].index
    if len(compounds_in_class) == 0:
        return pd.DataFrame()

    class_shap = (df_genes
                  .loc[df_genes.index.isin(compounds_in_class), gene_cols]
                  .mean(axis=0)
                  .dropna()
                  .sort_values(ascending=False)
                  .head(top_n))

    result = pd.DataFrame({
        "feature":        class_shap.index,
        "mean_abs_shap":  class_shap.values,
    })
    # Extract clean gene name and layer
    result["omics_layer"] = result["feature"].apply(
        lambda f: f.split("::", 1)[0] if "::" in f else "unknown"
    )
    result["gene_name"] = result["feature"].apply(
        lambda f: f.split("::", 1)[1] if "::" in f else f
    )
    result["n_compounds_in_class"] = len(compounds_in_class)
    return result[["gene_name", "omics_layer", "mean_abs_shap",
                   "n_compounds_in_class"]]

In [ ]:
# Compute top genes per MOA class
moa_classes = sorted(df_shap_layers["moa_class"].unique())
moa_top_genes: dict[str, pd.DataFrame] = {}

print(f"Top {TOP_N_GENES_PER_MOA} genes per MOA class:")
for moa_class in moa_classes:
    top = top_genes_for_class(moa_class, df_shap_genes, df_shap_layers)
    moa_top_genes[moa_class] = top
    n_compounds = (df_shap_layers["moa_class"] == moa_class).sum()
    print(f"\n[{moa_class}] - {n_compounds} compounds")
    print(top.head(10).to_string(index=False))

# Save CSV per class (input for KEGG/GO)
genes_dir = SHAP_DIR / "top_genes_per_moa"
genes_dir.mkdir(exist_ok=True)

for moa_class, df_genes_cls in moa_top_genes.items():
    fname = re.sub(r"[^a-zA-Z0-9_]", "_", moa_class) + ".csv"
    df_genes_cls.to_csv(genes_dir / fname, index=False)

print(f"\nSaved {len(moa_top_genes)} gene lists to: {genes_dir}")

In [ ]:
# Summary gene table - all MOA classes together
all_top_genes = pd.concat(
    [df_g.assign(moa_class=cls) for cls, df_g in moa_top_genes.items()],
    ignore_index=True,
)
all_top_genes.to_csv(SHAP_DIR / "all_top_genes_by_moa.csv", index=False)
print(f"Combined table: {SHAP_DIR / 'all_top_genes_by_moa.csv'}")
all_top_genes.head(15)

### C.4 Top-gene bar chart per MOA class

In [ ]:
# One subplot per MOA class (top 15 genes)
n_classes = len(moa_classes)
n_cols_plot = min(3, n_classes)
n_rows_plot = (n_classes + n_cols_plot - 1) // n_cols_plot

fig, axes = plt.subplots(n_rows_plot, n_cols_plot,
                          figsize=(6 * n_cols_plot, 4 * n_rows_plot))
axes_flat = np.array(axes).flatten() if n_classes > 1 else [axes]

LAYER_COLORS = {
    "expr": "#2c7bb6", "mut": "#d7191c",
    "cnv":  "#1a9641", "meth": "#fdae61", "prot": "#756bb1",
    "unknown": "#bdbdbd",
}

for ax, moa_class in zip(axes_flat, moa_classes):
    top = moa_top_genes[moa_class].head(15)
    if top.empty:
        ax.axis("off")
        continue
    colors = [LAYER_COLORS.get(l, "#bdbdbd") for l in top["omics_layer"]]
    bars = ax.barh(top["gene_name"][::-1],
                   top["mean_abs_shap"][::-1],
                   color=colors[::-1])
    ax.set_xlabel("Mean |SHAP|", fontsize=9)
    n_comp = top["n_compounds_in_class"].iloc[0]
    ax.set_title(f"{moa_class}\n(n={n_comp})", fontsize=10, fontweight="bold")
    ax.tick_params(axis="y", labelsize=8)

# Turn off unused axes
for ax in axes_flat[len(moa_classes):]:
    ax.axis("off")

# Legend for layers
handles = [plt.Rectangle((0, 0), 1, 1, fc=LAYER_COLORS.get(l, "#bdbdbd"),
                           label=l.upper())
           for l in ACTIVE_LAYERS]
fig.legend(handles=handles, title="Omics layer",
           loc="lower right", bbox_to_anchor=(0.98, 0.02), fontsize=9)

plt.suptitle(f"Top genes per MOA class (mean |SHAP|, {SHAP_MODE} mode)",
             fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(SHAP_DIR / "top_genes_per_moa_class.png",
            dpi=150, bbox_inches="tight")
plt.show()

### C.5 SHAP summary plot for one compound

Visualize raw SHAP values (direction + magnitude) for the top-scoring compound in the most populous MOA class.

In [ ]:
# Pick a compound: top Pearson in the most common non-Other MOA class
main_classes = [c for c in moa_classes if c != "Other/Unknown"]
if not main_classes:
    main_classes = moa_classes

In [ ]:
# Find compound with highest layer-importance signal in a well-represented class
target_class = class_sizes[main_classes].idxmax()
compounds_in_target = df_shap_layers[
    df_shap_layers["moa_class"] == target_class
].index.tolist()

In [ ]:
# Use first available compound with fold1 model
example_compound = None
for c in compounds_in_target:
    if load_fold_model(BEST_MODEL_FAMILY, c, BEST_STRATEGY, 1) is not None:
        example_compound = c
        break

In [ ]:
if example_compound is None:
    print("No compound found for summary plot - skip.")
else:
    print(f"Example compound: {example_compound}  (MOA: {target_class})")

    primary_layer = moa_layer_df.loc[target_class].idxmax()

    inference_key = BEST_STRATEGY if BEST_STRATEGY in feature_sets else "early_fusion"
    X_layer = feature_sets[inference_key]

    feat_names = X_layer.columns.to_numpy()
    n_feats = len(feat_names)

    y_comp = drug_response[example_compound].dropna()
    valid_ids = y_comp.index.to_numpy()

    model_ex = load_fold_model(BEST_MODEL_FAMILY, example_compound,
                                BEST_STRATEGY, 1)
    if model_ex is not None:
        from sklearn.model_selection import KFold
        kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
        splits = list(kf.split(valid_ids))
        train_idx, _ = splits[0]
        train_ids = valid_ids[train_idx]

        X_tr_np  = X_layer.loc[train_ids].to_numpy(dtype=np.float32)
        X_all_np = X_layer.loc[valid_ids].to_numpy(dtype=np.float32)

        try:
            sv = compute_shap_values(model_ex, X_tr_np, X_all_np, n_feats)

            if "::" in feat_names[0]:  # early_fusion (prefixed features)
                layer_mask = np.array([f.startswith(f"{primary_layer}::")
                                       for f in feat_names])
                sv_layer   = sv[:, layer_mask]
                X_vis      = X_layer.loc[valid_ids].iloc[:, layer_mask]
                plot_title = (f"SHAP summary - {example_compound}\n"
                              f"({BEST_MODEL_FAMILY}, {primary_layer} "
                              f"features from {inference_key} model)")
            else:  # single-omics
                sv_layer = sv
                X_vis    = X_layer.loc[valid_ids]
                plot_title = (f"SHAP summary - {example_compound}\n"
                              f"({BEST_MODEL_FAMILY}, {primary_layer} layer)")

            top20_idx = np.abs(sv_layer).mean(axis=0).argsort()[-20:][::-1]

            shap.summary_plot(
                sv_layer[:, top20_idx],
                X_vis.iloc[:, top20_idx],
                plot_type="dot",
                show=False,
                max_display=20,
            )
            plt.title(plot_title, fontsize=11)
            plt.tight_layout()
            plt.savefig(SHAP_DIR / f"shap_summary_{safe_compound_name(example_compound)}.png",
                        dpi=150, bbox_inches="tight")
            plt.show()
            print("Summary plot saved.")
        except Exception as e:
            print(f"Summary plot failed: {e}")

## Summary

In [ ]:
print("SHAP + MOA Analysis - Summary")
print(f"\nMOA annotation:")
print(f"  Total CTRPv2 compounds:     {len(df_moa)}")
print(f"  Annotated via ChEMBL:       {df_moa['mechanism_of_action'].notna().sum()}")
print(f"  MOA classes:                {df_moa['moa_class'].nunique()}")

print(f"\nSHAP analysis:")
print(f"  Mode:                       {SHAP_MODE}")
print(f"  Model:                      {BEST_MODEL_FAMILY} / {BEST_STRATEGY}")
print(f"  Compounds with SHAP:        {len(df_shap_layers)}")
print(f"  Active layers:              {ACTIVE_LAYERS}")

print(f"\nOutputs saved to {SHAP_DIR}:")
for p in sorted(SHAP_DIR.rglob("*")):
    if p.is_file():
        size_kb = p.stat().st_size / 1024
        print(f"  {p.relative_to(SHAP_DIR)}  ({size_kb:.0f} KB)")

In [ ]:
!cp -r /content/data/shap_results /content/drive/MyDrive/data